In [ ]:
from pathlib import Path
import pickle
import sys

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Resolve paths relative to this notebook location.
model_path = Path("pace_predictor_rf_final_partie2.pkl")
preprocessed_path = Path("../../Reinforcement_learning_partie/master_dataset_partie2_2024_stint_preprocessed.csv")
raw_2024_path = Path("../../classes/master_dataset_partie2_2024_stint.csv")
preprocess_module_dir = Path("../../Reinforcement_learning_partie").resolve()

print("Model path:", model_path.resolve())
print("RL preprocessed path:", preprocessed_path.resolve())
print("Raw 2024 stint path:", raw_2024_path.resolve())

# Load model and RL preprocessed features.
with open(model_path, "rb") as f:
    model = pickle.load(f)

df_rl = pd.read_csv(preprocessed_path)
print(f"\nRL preprocessed dataset shape: {df_rl.shape}")

# Validate feature compatibility for RL dataset.
expected_features = list(getattr(model, "feature_names_in_", []))
if expected_features:
    missing_features = [c for c in expected_features if c not in df_rl.columns]
    extra_features = [c for c in df_rl.columns if c not in expected_features]
else:
    expected_features = list(df_rl.columns)
    missing_features = []
    extra_features = []

print("\n=== RL Feature Compatibility ===")
print("Expected by model:", len(expected_features))
print("Missing features:", missing_features)
print("Extra features:", extra_features)

# Sanity predictions on RL preprocessed dataset.
X_rl = df_rl[expected_features].copy()
na_counts_rl = X_rl.isna().sum()
na_cols_rl = na_counts_rl[na_counts_rl > 0].sort_values(ascending=False)

if na_cols_rl.empty:
    print("\nNo missing values in RL inference features.")
    X_rl_safe = X_rl
else:
    print("\nMissing values in RL inference features:")
    print(na_cols_rl)
    X_rl_safe = X_rl.fillna(X_rl.median(numeric_only=True))

pred_rl = pd.Series(model.predict(X_rl_safe), name="pred_delta_next_lap")
print("\n=== RL Prediction Sanity ===")
print(pred_rl.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))
print("Predictions < 0:", int((pred_rl < 0).sum()))
print("Predictions > 3:", int((pred_rl > 3).sum()))

# -----------------------------------------------------------------------------
# Recalculate MAE on labeled data by rebuilding target with RL preprocessing.
# -----------------------------------------------------------------------------
sys.path.append(str(preprocess_module_dir))
from preprocess import preprocess_data, _postprocess_features, _get_model_features  # noqa: E402

df_raw = pd.read_csv(raw_2024_path)
df_proc = preprocess_data(df_raw)
df_proc = _postprocess_features(df_proc)

if "delta_next_lap" not in df_proc.columns:
    raise ValueError("delta_next_lap is missing after preprocessing. Cannot compute MAE.")

feature_cols = _get_model_features(df_proc)
X_labeled = df_proc[feature_cols].copy()
y_true = df_proc["delta_next_lap"].copy()

# Align to model features order if available.
if expected_features:
    X_labeled = X_labeled[expected_features]

# Handle any remaining NaN before prediction.
X_labeled_safe = X_labeled.fillna(X_labeled.median(numeric_only=True))

y_pred = model.predict(X_labeled_safe)

mae = mean_absolute_error(y_true, y_pred)
rmse = mean_squared_error(y_true, y_pred) ** 0.5
r2 = r2_score(y_true, y_pred)

print("\n=== Recalculated Metrics (labeled 2024 stint) ===")
print(f"Rows used: {len(y_true)}")
print(f"MAE:  {mae:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"R2:   {r2:.6f}")

# Optional consistency check: RL preprocessed file vs rebuilt feature table shape.
print("\n=== Consistency Check ===")
print("RL preprocessed rows:", len(df_rl))
print("Rebuilt labeled rows:", len(df_proc))
if len(df_rl) != len(df_proc):
    print("Note: row counts differ. MAE is computed on the rebuilt labeled dataset.")

Model path: C:\Users\user\Desktop\personal\F1_strategy(ejjaw yebda hna ya zamil\Models.ipynb\models\pace_predictor_rf_final_partie2.pkl
Dataset path: C:\Users\user\Desktop\personal\F1_strategy(ejjaw yebda hna ya zamil\Reinforcement_learning_partie\master_dataset_partie2_2024_stint_preprocessed.csv

Dataset shape: (17799, 21)
Columns (21): ['CompoundEncoded', 'TyreLife', 'TrackTemp', 'FuelLoad', 'Abrasivity', 'LateralEnergy', 'DeltaToBest', 'LapNumber', 'Stint', 'RaceNumber', 'TeamEncoded', 'delta_velocity', 'lateral_stress_cumul', 'abrasive_stress_cumul', 'stress_x_temp', 'compound_x_abrasivity', 'compound_x_lateral', 'compound_x_tyrelife', 'prev_stint_max_delta', 'stint_length', 'tyre_life_pct']

=== Feature Compatibility ===
Expected by model: 21
Missing features: []
Extra features: []

=== Missing Values ===
prev_stint_max_delta    5988
dtype: int64

=== Prediction Sanity Check ===
count    17799.000000
mean         0.595202
std          0.459645
min          0.000799
1%           0